<a href="https://colab.research.google.com/github/Innovatewithapple/transformer-paper-rag/blob/main/transformer_paper_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers faiss-cpu pymupdf

In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import re
import faiss
from sentence_transformers import SentenceTransformer,CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM,pipeline
import pymupdf
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from huggingface_hub import login
login(token="")  # your token here

In [ ]:
#----------Load Encoder Model------------#
encoder_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)

#---------Load Decoder Models-----------#
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token  # ← add this
llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",dtype=torch.float16,device_map="auto",trust_remote_code=True)

In [ ]:
# ---- LOAD RERANKER (once at startup) ----
# swap model name here anytime — nothing else changes
reranking_model = CrossEncoder('BAAI/bge-reranker-large',device='cuda:0')

In [ ]:

print(decoder_tokenizer.chat_template)

In [ ]:
#------Extract and Clean----------!
def extract_and_clean_pdf(pdf_path):
  doc = pymupdf.open(pdf_path)
  full_text = ""
  for page in doc:
    full_text += page.get_text() + "\n"

  full_text = re.sub(r'-\n', '', full_text) #Removes broken hyphenated words across lines. (transf-ormer = transformer)
  full_text = re.sub(r'\n\d+\n', '\n', full_text) #Removes page numbers.
  full_text = re.sub(r'\[\d+\]', '', full_text) #Removes citation-like patterns.
  full_text = re.sub(r'\n{3,}', '\n\n', full_text) #Removes excessive empty lines.
  full_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', full_text) #This fixes broken sentences. next char or previous char shouldn't be in next line
  full_text = re.sub(r' {2,}', ' ', full_text) #Removes multiple spaces.
  # remove email addresses
  full_text = re.sub(r'\S+@\S+\.\S+', '', full_text)

  # remove lines that are only names/affiliations (short lines before abstract)
  # everything before "Abstract" is likely title page noise
  abstract_pos = full_text.find('Abstract')
  if abstract_pos != -1:
      full_text = full_text[abstract_pos:]  # start from Abstract

  return full_text.strip()

In [ ]:
# ---- SANITY CHECK ----
text = extract_and_clean_pdf('/kaggle/input/datasets/mihirvyas/attentionisallyouneedpaper/NIPS-2017-attention-is-all-you-need-Paper.pdf')
sentences = sent_tokenize(text)
avg_sent = sum(len(s) for s in sentences) // len(sentences)
max_sent = max(len(s) for s in sentences)
min_sent = min(len(s) for s in sentences)

# model limits
embedding_token_limit = 8192
embedding_safe_chars  = embedding_token_limit * 4

llm_token_limit = 32768  # Qwen2.5
top_k = 3
child_size  = 400
parent_size = 1200

print("====== SANITY CHECK ======")
print(f"\n--- DOCUMENT ---")
print(f"Total chars:         {len(text)}")
print(f"Total sentences:     {len(sentences)}")
print(f"Avg sentence length: {avg_sent} chars")
print(f"Max sentence length: {max_sent} chars")
print(f"Min sentence length: {min_sent} chars")

print(f"\n--- EMBEDDING MODEL ---")
print(f"Token limit:         {embedding_token_limit}")
print(f"Safe chunk limit:    {embedding_safe_chars} chars")
print(f"Your child_size:     {child_size} chars")
if child_size < embedding_safe_chars:
    print(f"Status:              ✓ SAFE")
else:
    print(f"Status:              ✗ DANGER")

print(f"\n--- LLM CONTEXT ---")
tokens_to_llm = (top_k * parent_size) // 4
print(f"Tokens sent to LLM:  {tokens_to_llm}")
print(f"LLM token limit:     {llm_token_limit}")
if tokens_to_llm < llm_token_limit * 0.7:
    print(f"Status:              ✓ SAFE")
else:
    print(f"Status:              ✗ DANGER")

print(f"\n--- CHUNK ESTIMATE ---")
expected_chunks = len(text) // child_size
print(f"Expected chunks:     {expected_chunks}")
print(f"If actual >> this:   chunking logic broken")
print("==========================")

In [ ]:
def parent_child_chunk(text,parent_size=1200,child_size=500,overlap=1):
  sentence = sent_tokenize(text)

  #build parents
  parents = []
  current = []
  current_len = 0

  for sent in sentence:
    if current_len + len(sent) < parent_size:
      current.append(sent)
      current_len += len(sent)
    else:
      if current:
        parents.append(" ".join(current))
      current = current[-overlap:] + [sent]
      current_len = sum(len(s) for s in current)

  if current:
    parents.append(" ".join(current))


  #build children
  result = []
  for p_idx,parent in enumerate(parents):
    children = []
    current = []
    current_len = 0

    for sent in sent_tokenize(parent):
      if current_len + len(sent) < child_size:
        current.append(sent)
        current_len += len(sent)
      else:
        if current:
          children.append(" ".join(current))
        current = current[-overlap:] + [sent]
        current_len = sum(len(s) for s in current)

    if current:
      children.append(" ".join(current))

    for c_idx,child in enumerate(children):
      result.append({
          'child_text':child,
          'parent_text':parent,
          'p_idx':p_idx,
          'child_idx':f'{p_idx}_{c_idx}'
      })

  return result

In [ ]:
text = extract_and_clean_pdf('/kaggle/input/datasets/mihirvyas/attentionisallyouneedpaper/NIPS-2017-attention-is-all-you-need-Paper.pdf')
All_chunks = parent_child_chunk(text=text,parent_size=1200,child_size=600,overlap=1)
print(len(All_chunks))

98


In [ ]:
#--------Extract child texts for embedding--------!
child_texts = [c['child_text'] for c in All_chunks]
parent_texts = [c['parent_text'] for c in All_chunks]
p_idx = [c['p_idx'] for c in All_chunks]
child_idx = [c['child_idx'] for c in All_chunks]

prefix = ['search_document: '+t for t in child_texts]

embeddings = encoder_model.encode(prefix,batch_size=32,show_progress_bar=True,normalize_embeddings=True)

#-------Build faiss index--------!
dim = embeddings.shape[1] # 786 features per vector

index = faiss.IndexFlatIP(dim) #Index is a vectordb structure inside faiss, and flat just flatten the features and apply dot product/cosine similarity
index.add(embeddings.astype(np.float32))
print(f'fiass index build. total vectors: {index.ntotal}')

In [ ]:
#---------Retrieval Method--------!
def Retrieval(question,top_k=5):
  query = ['search_query: '+question]
  query_vector = encoder_model.encode(query,normalize_embeddings=True).astype(np.float32)

  #Search
  scores,indices = index.search(query_vector,top_k)
  results = []

  for score,idx in zip(scores[0],indices[0]):
    results.append({
        'score':float(score),
        'child_text':child_texts[idx],
        'parent_text':parent_texts[idx],
        'child_id':child_idx[idx]
    })
      # clean printing

  # for i, r in enumerate(results):
  #    print("=" * 80)
  #    print(f"Rank      : {i+1}")
  #    print(f"Score     : {r['score']:.4f}")
  #    print(f"Child ID  : {r['child_id']}")

  #    print("\nChild Text:\n")
  #    print(r['child_text'])

  #    print("\nParent Text:\n")
  #    print(r['parent_text'])

  #    print("\n")
  return results

# **CrossEncoder (Reranking)**

In [ ]:
def Reranking(question,results,top_k=2):
    pairs = [(question,r['child_text']) for r in results]
    scores = reranking_model.predict(pairs)
    # print('RetrievedScores: ',scores)

    for i,result in enumerate(results):
        result['rerank_score'] = float(scores[i])

    #Put best scores first
    reranked = sorted(results,key=lambda x:x['rerank_score'],reverse=True) # put true makes high scores first, default put low scores first
    # for r in reranked:
    #  print("="*80)
    #  print("Question: ",question)
    #  print("Rerank Score:", r['rerank_score'])
    #  print("\nText:\n")
    #  print(r['child_text'])
    return reranked[:top_k]

# **Decoder**

In [ ]:
#-------Prompt Builder-------!
def buildPrompt(question,result):
  context = "\n---\n".join([r['child_text'] for r in result])

  message = [
      {"role": "system", "content": """
You are a precise question-answering assistant.

Answer using ONLY the provided context.

Rules:
1. Do NOT use outside knowledge.
2. Copy numerical and decimal values exactly as written in the context.
3. Do NOT round, estimate, modify, or infer numbers.
4. Do NOT explain or expand abbreviations unless explicitly written in the context.
5. Preserve important entities, task names, benchmark names, and relationships exactly as written in the context.
6. Include the specific task or subject associated with the answer if present in the context.
7. Give a clear and natural answer in 1-2 sentences.
8. Keep the answer grounded strictly in the provided context.
9. If the exact answer is not explicitly stated but can be reasonably inferred from the context, provide the most likely answer based ONLY on the context.
10.If the context does not contain enough information, say:
    "I don't know."
"""},
      {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
  ]

  prompt = decoder_tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
  return prompt

In [ ]:
def Answer(question):
  result = Retrieval(question,top_k=3)
  top_3 = Reranking(question,result,top_k=1)
  prompt = buildPrompt(question,top_3)

  inputs = decoder_tokenizer(prompt,return_tensors='pt').to(llm.device)
  input_tokens = inputs['input_ids'].shape[1] # input has batch size and token count, we need this count to get answer without question include in text

  #generate
  with torch.no_grad():
    output = llm.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.1,
        repetition_penalty=1.0,
        no_repeat_ngram_size=3,
        top_p=0.9,
        pad_token_id=decoder_tokenizer.eos_token_id
    )

  # decode only the answer part
  answer_tokens = output[0][input_tokens:]
  return decoder_tokenizer.decode(answer_tokens, skip_special_tokens=True)

**Question-Answers******

In [ ]:
questions = [
    "What model architecture is introduced in the paper?",
    "What does the encoder in Transformer consist of?",
    "How many attention heads did they use?",
    "What BLEU score did they achieve on English-German translation?",
    "How long did training take on 8 GPUs?",
    "What is the key advantage over RNN models?",
    "What optimizer did they use?",
    "What is multi-head attention?"
]

In [ ]:
#------Retrieve--------!
for q in questions:
    print(f"Q: {q}")
    result = Answer(q)
    print(f"\nA: {result}")
    print("---\n")


Q: What model architecture is introduced in the paper?

A: The Transformer model architecture, which eschews recurrence and relies entirely on attention mechanisms to draw dependencies between inputs and outputs.
---

Q: What does the encoder in Transformer consist of?

A: The encoder in the Transformer model consists of a stacked structure of 6 layers, each containing two sublayers: a multi-headed self-Attention mechanism and a simple fully connected, position-wise feed-forward neural network.
---

Q: How many attention heads did they use?

A: They used 8 attention heads.
---

Q: What BLEU score did they achieve on English-German translation?

A: The big transformer (Transformer) model achieved a new benchmark BLEU (Bilingual Evaluation Understudy) score of "28.40" on the English-to- German translation task.
---

Q: How long did training take on 8 GPUs?

A: Training took three point five days on eight P100s.
---

Q: What is the key advantage over RNN models?

A: The key advantage of s

In [ ]:
question = "What optimizer did they use?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

Q: What optimizer did they use?
Rank      : 1
Score     : 0.7023
Child ID  : 20_1

Child Text:

5.3 Optimizer We used the Adam optimizer with β1 = 0.9, β2 = 0.98 and ϵ = 10−9. We varied the learning rate over the course of training, according to the formula: lrate = d−0.5 model · min(step_num−0.5, step_num · warmup_steps−1.5) (3) This corresponds to increasing the learning rate linearly for the ﬁrst warmup_steps training steps, and decreasing it thereafter proportionally to the inverse square root of the step number. We used warmup_steps = 4000.

Parent Text:

5.2 Hardware and Schedule We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using the hyperparameters described throughout the paper, each training step took about 0.4 seconds. We trained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the bottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps (3.5 days). 5.3 Opti

In [ ]:
question = "What BLEU score did they achieve on English-German translation?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

Q: What BLEU score did they achieve on English-German translation?
Rank      : 1
Score     : 0.7607
Child ID  : 22_2

Child Text:

This hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score. Results 6.1 Machine Translation On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big) in Table 2) outperforms the best previously reported models (including ensembles) by more than 2.0 BLEU, establishing a new state-of-the-art BLEU score of 28.4. The conﬁguration of this model is listed in the bottom line of Table 3. Training took 3.5 days on 8 P100 GPUs.

Parent Text:

Model BLEU Training Cost (FLOPs) EN-DE EN-FR EN-DE EN-FR ByteNet 23.75 Deep-Att + PosUnk 39.2 1.0 · 1020 GNMT + RL 24.6 39.92 2.3 · 1019 1.4 · 1020 ConvS2S 25.16 40.46 9.6 · 1018 1.5 · 1020 MoE 26.03 40.56 2.0 · 1019 1.2 · 1020 Deep-Att + PosUnk Ensemble 40.4 8.0 · 1020 GNMT + RL Ensemble 26.30 41.16 1.8 · 1020 1.1 · 1021 ConvS2S Ensemble 26.36 41.29 

In [ ]:
question = "What does the encoder in Transformer consist of?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

Q: What does the encoder in Transformer consist of?
Rank      : 1
Score     : 0.7889
Child ID  : 6_2

Child Text:

At each step the model is auto-regressive , consuming the previously generated symbols as additional input when generating the next. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers.

Parent Text:

To the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying entirely on self-attention to compute representations of its input and output without using sequencealigned RNNs or convolution. In the following sections, we will describe the Transformer, motivate self-attention and discuss its advantages over models such as [14, 15] and . Model Architecture Most competitive neural sequence

In [ ]:
question = "How many attention heads did they use?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

Q: How many attention heads did they use?
Rank      : 1
Score     : 0.7251
Child ID  : 11_1

Child Text:

In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder.

Parent Text:

With a single attention head, averaging inhibits this. 4To illustrate why the dot products get large, assume that the components of q and k are independent random variables with mean 0 and variance 1. Then their dot product, q · k = Pdk i=1 qiki, has mean 0 and variance dk. MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O where headi = Attention(

In [ ]:
question = "How long did training take on 8 GPUs?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

Q: How long did training take on 8 GPUs?
Rank      : 1
Score     : 0.8702
Child ID  : 22_3

Child Text:

Training took 3.5 days on 8 P100 GPUs. Even our base model surpasses all previously published models and ensembles, at a fraction of the training cost of any of the competitive models.

Parent Text:

Model BLEU Training Cost (FLOPs) EN-DE EN-FR EN-DE EN-FR ByteNet 23.75 Deep-Att + PosUnk 39.2 1.0 · 1020 GNMT + RL 24.6 39.92 2.3 · 1019 1.4 · 1020 ConvS2S 25.16 40.46 9.6 · 1018 1.5 · 1020 MoE 26.03 40.56 2.0 · 1019 1.2 · 1020 Deep-Att + PosUnk Ensemble 40.4 8.0 · 1020 GNMT + RL Ensemble 26.30 41.16 1.8 · 1020 1.1 · 1021 ConvS2S Ensemble 26.36 41.29 7.7 · 1019 1.2 · 1021 Transformer (base model) 27.3 38.1 3.3 · 1018 Transformer (big) 28.4 41.0 2.3 · 1019 Label Smoothing During training, we employed label smoothing of value ϵls = 0.1 . This hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score. Results 6.1 Machine Translation On the WMT 2014 Engli

In [ ]:
question = "What is the key advantage over RNN models?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

Q: What is the key advantage over RNN models?
Rank      : 1
Score     : 0.6779
Child ID  : 1_1

Child Text:

Introduction Recurrent neural networks, long short-term memory and gated recurrent neural networks in particular, have been ﬁrmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [29, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [31, 21, 13]. ∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started the effort to evaluate this idea.

Parent Text:

On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.0 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. Introduction Recurrent neural networks, long short-term memory 

In [ ]:
question = "What model architecture is introduced in the paper?"
print(f"Q: {question}")
result = Answer(question)
print(f"\nA: {result}")
print("---\n")

Q: What model architecture is introduced in the paper?
Rank      : 1
Score     : 0.6657
Child ID  : 6_1

Child Text:

Model Architecture Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29]. Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence of continuous representations z = (z1, ..., zn). Given z, the decoder then generates an output sequence (y1, ..., ym) of symbols one element at a time. At each step the model is auto-regressive , consuming the previously generated symbols as additional input when generating the next.

Parent Text:

To the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying entirely on self-attention to compute representations of its input and output without using sequencealigned RNNs or convolution. In the following sections, we will describe the Transformer, motivate self-attention and discuss its advantages over models such as [14, 15] and .